# seq2seq — Sequence to Sequence Learning with Neural Networks (2014)

_Papers · Sequence & NLP_

**Read a whole input sequence into one vector with an LSTM, then generate the output sequence with another LSTM.**

---

We build seq2seq from the ground up: first we open up a single LSTM cell to see its gates, then we wire an encoder LSTM to a decoder LSTM and train it on a copy task — checking the paper's source-reversal trick along the way. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Open up one LSTM cell and confirm its gates

seq2seq is built from LSTMs, so first we make sure we understand a single LSTM step. An LSTM mixes the input `x` and previous hidden state `h0` through one big linear layer, then splits the result into four **gates**: input `i`, forget `f`, candidate `g`, and output `o`. We compute those gates by hand and check they match PyTorch's own `nn.LSTMCell`, then we redo the same arithmetic as plain scalars so the numbers are fully transparent.

In [ ]:
import torch
import torch.nn as nn
import math

# ===== one nn.LSTMCell step == hand-written i/f/g/o gates =====
torch.manual_seed(7)
cell = nn.LSTMCell(2, 2)

x = torch.tensor([[1.0, -1.0]])
h0 = torch.zeros(1, 2)
c0 = torch.zeros(1, 2)

# The single linear pre-activation, then split into the four gates.
gates = x @ cell.weight_ih.t() + h0 @ cell.weight_hh.t() + cell.bias_ih + cell.bias_hh
i_, f_, g_, o_ = gates.chunk(4, 1)            # PyTorch packs gates as [i, f, g, o]

# Apply each gate's nonlinearity.
i_ = torch.sigmoid(i_)
f_ = torch.sigmoid(f_)
g_ = torch.tanh(g_)
o_ = torch.sigmoid(o_)

c1 = f_ * c0 + i_ * g_                         # cell update
h1 = o_ * torch.tanh(c1)                       # hidden update

h_ref, c_ref = cell(x, (h0, c0))              # PyTorch's own LSTM step
print("LSTM cell allclose h:", torch.allclose(h1, h_ref, atol=1e-6),
      "| c:", torch.allclose(c1, c_ref, atol=1e-6))   # True True

# ===== recompute the WORKED scalar example (h_{t-1}=c_{t-1}=0, x=1) =====
i = 1 / (1 + math.exp(-0.5))
f = 1 / (1 + math.exp(-1.0))
g = math.tanh(1.0)
o = 1 / (1 + math.exp(-0.5))

c = f * 0 + i * g
h = o * math.tanh(c)
print(f"worked  i={i:.5f} f={f:.5f} g={g:.5f} o={o:.5f}")   # 0.62246 0.73106 0.76159 0.62246
print(f"worked  c={c:.5f} h={h:.5f}")                       # c=0.47406 h=0.27480

### Step 2 — Wire an encoder LSTM to a decoder LSTM

Now the model itself. The **encoder** LSTM reads the whole source sequence and squeezes it into one context vector `v` — its final `(hidden, cell)` state. The **decoder** is a *separate* LSTM that is seeded with `v` and generates the output one token at a time. We train it on a tiny **copy task** (output should equal input) with teacher forcing, which feeds the true previous token as the next decoder input. The `reverse_input` flag implements the paper's Section 3.3 trick of flipping the source.

In [ ]:
V, BOS, L, EMB, HID = 12, 10, 12, 20, 40       # vocab 0..9 + <bos>=10 + <eos>=11

def make_batch(n):
    src = torch.randint(0, 10, (n, L))
    tgt = src.clone()                          # COPY task: output == input
    return src, tgt

class Seq2Seq(nn.Module):
    def __init__(self, reverse_input):
        super().__init__()
        self.reverse_input = reverse_input
        self.emb = nn.Embedding(V, EMB)
        self.enc = nn.LSTM(EMB, HID, batch_first=True)   # encoder LSTM
        self.dec = nn.LSTM(EMB, HID, batch_first=True)   # decoder LSTM (separate weights)
        self.out = nn.Linear(HID, V)

    def forward(self, src, tgt_in):
        if self.reverse_input:
            src = torch.flip(src, dims=[1])             # PAPER TRICK: reverse the source
        _, (h, c) = self.enc(self.emb(src))             # context v = final (hidden, cell)
        dec_out, _ = self.dec(self.emb(tgt_in), (h, c)) # decoder seeded from v
        return self.out(dec_out)                        # (n, L, V) logits

def train(reverse_input, steps=500, seed=0):
    torch.manual_seed(seed)
    m = Seq2Seq(reverse_input)
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    lossf = nn.CrossEntropyLoss()
    for s in range(steps):
        src, tgt = make_batch(64)
        bos = torch.full((64, 1), BOS)
        tgt_in = torch.cat([bos, tgt[:, :-1]], dim=1)   # teacher forcing
        loss = lossf(m(src, tgt_in).reshape(-1, V), tgt.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        src, tgt = make_batch(500)
        bos = torch.full((500, 1), BOS)
        tgt_in = torch.cat([bos, tgt[:, :-1]], dim=1)
        acc = (m(src, tgt_in).argmax(-1) == tgt).float().mean().item()
    return m, acc

### Step 3 — Ablation: does reversing the source help?

The paper's headline trick is to feed the source sequence **backwards**. We test it directly: train five models with `reverse_input=True` and five with `reverse_input=False`, then compare the average token accuracy. Reversing shortens the gap between each source token and the output token that copies it, so the LSTM's gradients have a shorter path to travel — and the reversed runs should win.

In [ ]:
accT = [train(True,  500, s)[1] for s in range(5)]
accF = [train(False, 500, s)[1] for s in range(5)]
print("token acc reverse=True :", round(sum(accT) / 5, 3), [round(a, 3) for a in accT])
print("token acc reverse=False:", round(sum(accF) / 5, 3), [round(a, 3) for a in accF])
#   -> reverse=True ~0.795  >  reverse=False ~0.544  (OUR toy run, not the paper's BLEU)

### Step 4 — Decode autoregressively (free-running)

Teacher forcing fed the true previous token during training. At real inference time there is no ground truth, so the decoder must feed its **own** previous output back in. We encode three sources, seed the decoder from the context vector, and roll forward one token at a time. The earliest output positions copy cleanly; later positions are harder for a one-layer toy net.

In [ ]:
m, _ = train(True, 600, 0)
m.eval()
with torch.no_grad():
    src, _ = make_batch(3)
    enc_in = torch.flip(src, dims=[1])
    _, (h, c) = m.enc(m.emb(enc_in))

    cur = torch.full((3, 1), BOS)
    outs = []
    for t in range(L):                          # feed model's own output back in
        d, (h, c) = m.dec(m.emb(cur), (h, c))
        cur = m.out(d[:, -1]).argmax(-1, keepdim=True)
        outs.append(cur)
    gen = torch.cat(outs, 1)

for i in range(3):
    print("src", src[i].tolist(), "-> gen", gen[i].tolist())
# the first (freshest) tokens copy correctly; longer positions are harder on a 1-layer toy net

## Visualize the data & results

_On a toy copy task, does reversing the source sequence (the paper's Section 3.3 trick) make the seq2seq LSTM learn faster and end more accurate than feeding it in normal order?_

### Step 1 — Rebuild a compact model and a loss-recording loop

To plot the learning curves we need the loss *over time*, so this self-contained block re-defines the same `Seq2Seq` model and a `curve` helper that trains it while recording the loss every 50 steps. It is the same architecture as above, just packaged to return a list of `(step, loss)` points.

In [ ]:
import torch
import torch.nn as nn

V, BOS, L, EMB, HID = 12, 10, 12, 20, 40

def make_batch(n):
    src = torch.randint(0, 10, (n, L))
    return src, src.clone()                     # copy task

class Seq2Seq(nn.Module):
    def __init__(self, rev):
        super().__init__()
        self.rev = rev
        self.emb = nn.Embedding(V, EMB)
        self.enc = nn.LSTM(EMB, HID, batch_first=True)
        self.dec = nn.LSTM(EMB, HID, batch_first=True)
        self.out = nn.Linear(HID, V)

    def forward(self, src, ti):
        if self.rev:
            src = torch.flip(src, dims=[1])
        _, (h, c) = self.enc(self.emb(src))
        d, _ = self.dec(self.emb(ti), (h, c))
        return self.out(d)

def curve(rev, steps=500, seed=0):
    torch.manual_seed(seed)
    m = Seq2Seq(rev)
    opt = torch.optim.Adam(m.parameters(), 3e-3)
    lf = nn.CrossEntropyLoss()
    pts = []
    for s in range(steps):
        src, tgt = make_batch(64)
        bos = torch.full((64, 1), BOS)
        ti = torch.cat([bos, tgt[:, :-1]], 1)
        loss = lf(m(src, ti).reshape(-1, V), tgt.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if s % 50 == 0 or s == steps - 1:       # sample the loss for plotting
            pts.append((s, round(loss.item(), 4)))
    return pts

### Step 2 — Compare the two learning curves

Finally we run the recorder for both settings and print the sampled loss points. The `reverse=True` curve (green, above) should drop faster and end lower than the `reverse=False` curve (orange) — the same direction as the paper's reversal result, reproduced on toy data.

In [ ]:
print("reverse=True :", curve(True))     # the green curve plotted above
print("reverse=False:", curve(False))    # the orange curve plotted above

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Recompute the worked LSTM step from $h_{t-1}=0,\ c_{t-1}=0,\ x_t=1$ but with the forget-gate pre-activation changed to $-2$ (all other pre-activations the same: input $0.5$, candidate $1.0$, output $0.5$). What are $c_t$ and $h_t$, and why is $c_t$ unchanged from the original?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Gates: $i_t=\sigma(0.5)=0.62246$, $f_t=\sigma(-2)=0.11920$, $g_t=\tanh(1)=0.76159$, $o_t=\sigma(0.5)=0.62246$. — _Only the forget gate changed._
- $c_t=f_t\cdot c_{t-1}+i_t\cdot g_t = 0.11920\cdot 0 + 0.62246\cdot 0.76159 = 0.47406$. — _Since $c_{t-1}=0$, the forget gate multiplies zero — it has no effect on the first step._
- $h_t=o_t\cdot\tanh(c_t)=0.62246\cdot\tanh(0.47406)=0.27480$. — _Same output gate and same $c_t$ as the original._

**Answer:** $c_t=0.47406$, $h_t=0.27480$ — identical to the original worked example. The forget gate only scales the previous cell state $c_{t-1}$, which is $0$ here, so changing $f_t$ does nothing on the very first step. The forget gate only starts to matter once there is accumulated memory to forget.

</details>

**Problem 2.** For a 3-token output, Eq. 1 gives $p(y_1,y_2,y_3\mid x) = p(y_1\mid v)\,p(y_2\mid v,y_1)\,p(y_3\mid v,y_1,y_2)$. If the decoder assigns the correct tokens probabilities $0.5,\ 0.8,\ 0.25$, what is the sequence probability and the total negative log-likelihood (the training loss summed over steps)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Sequence probability $= 0.5\times 0.8\times 0.25 = 0.1$. — _Eq. 1 multiplies the per-step probabilities._
- Total NLL $= -\ln(0.5)-\ln(0.8)-\ln(0.25) = 0.6931+0.2231+1.3863 = 2.3026$. — _Cross-entropy loss is the negative log of each factor, summed (equivalently $-\ln 0.1$)._
- Check: $-\ln(0.1)=2.3026$. — _Sum of the logs equals the log of the product — same number._

**Answer:** Sequence probability $=0.1$; total negative log-likelihood $=2.3026$. Training minimizes this sum; because logs turn the product into a sum, the loss is just the per-token cross-entropies added up — exactly what nn.CrossEntropyLoss does across the output positions.

</details>

**Problem 3.** Ablation (from the CODE): on the toy copy task, flip reverse_input from True to False and retrain. Predict what happens to the loss curve and final token accuracy, and explain it using "time lag."

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set reverse_input=False and retrain with the same seed and steps. — _Isolates the source-reversal switch — the paper's Section 3.3 ablation._
- Compare the loss curves and the 5-seed mean token accuracy printed by the CODE. — _The qualitative gap is the reproducible signal, not a single number._
- Reason about distances: with reverse_input=True the first source token ends nearest the decoder start, so the first output (which depends on it) has a short dependency. — _That is the "minimal time lag" the paper describes._

**Answer:** In our run, reverse_input=True learns the copy faster and reaches higher token accuracy (mean ≈ 0.80 over 5 seeds) than reverse_input=False (mean ≈ 0.54) — see the CODEVIZ curves. Reversing the source shortens the time lag between corresponding input and output tokens, so the LSTM's gradients have an easier short-range path to learn. This reproduces the direction of the paper's Section 3.3 effect (where it raised test BLEU 25.9 → 30.6) on toy data — our numbers, not the paper's.

</details>